## Second-Highest Salary
### Accenture: Find the second-highest salary and compare approaches that scale.


#### Concept & Dataset adapted from: StrataScratch Data Projects (https://platform.stratascratch.com/)

In [2]:
#importing files
import pandas as pd
import numpy as np

In [14]:
#reading the data
data_raw = pd.read_csv("accenture_employees.csv")

#### Knowing data shape and type

In [11]:
print("rows,cols : ",data_raw.shape)

rows,cols :  (63, 5)


In [12]:
print(data_raw.dtypes)

employee_id     int64
full_name      object
title          object
salary         object
hire_date      object
dtype: object


#### Testing the Poteintial problem with the data

In [19]:
#problem 1: salary as string (why)
data_probe = pd.to_numeric(data_raw["salary"],errors = "coerce")
non_numeric_salary_rows= data_raw[(data_probe.isna() ) & (data_raw["salary"].notna())]

print("Number of Duplicated rows : ", data_raw.duplicated().sum())
print("Number of Rows with Blank salary : ", data_raw["salary"].isna().sum())
print("Number of non numeric salary rows : ",len(non_numeric_salary_rows))

Number of Duplicated rows :  0
Number of Rows with Blank salary :  5
Number of non numeric salary rows :  1


In [20]:
print(non_numeric_salary_rows)

   employee_id    full_name           title   salary   hire_date
1         4003  Anika Conti  Senior Manager  198,000  2016-11-18


##### From above ouput we can see that there is one row in salary column that converted the complete salary column to text as it is containing a comma.  Also, there are 5 blank rows in salary column and no duplicate records in the dataset  


In [22]:
# what happens when we use this uncleaned data to find the second highest salary
max_with_string_type_salary = data_raw["salary"].dropna().max()
print("Max salary with salary type as text : ", max_with_string_type_salary)

Max salary with salary type as text :  99000


##### The above output is wrong as the max is being calculated lexicographically because the datatype is text.                                          Now if we conver this data column to numeric, we can see other potential problem. 

In [23]:
num_salary = pd.to_numeric(data_raw["salary"], errors = "coerce")
second_max_on_num_sal = num_salary[num_salary<num_salary.max()].max()
print("second max salary after converting to numeric : ", second_max_on_num_sal)

second max salary after converting to numeric :  181000.0


##### This output is again wrong as the second highest salary is 198,000, which was silently removed due to coerce beacuse it had a comma as separator.

### Hence we need to clean the data first

In [27]:
data_clean = data_raw.assign(
    salary = pd.to_numeric(
        data_raw["salary"].str.replace(",","",regex=False)
    ).astype("Int64")
)

In [33]:
print("salary datatype now : ",data_clean["salary"].dtype,"\nnumber of blank salaries kept : ", data_clean["salary"].isna().sum())

salary datatype now :  Int64 
number of blank salaries kept :  5


In [43]:
#inspecting cleaned row
data_clean.loc[non_numeric_salary_rows.index, ['employee_id',"full_name","title","salary","hire_date"]]

,employee_id,full_name,title,salary,hire_date
1,4003,Anika Conti,Senior Manager,198000,2016-11-18


### Finding the salary distibution

In [45]:
data_clean["salary"].value_counts().sort_index(ascending = False).head()

salary
250000    2
198000    1
181000    1
180000    1
178000    2
Name: count, dtype: Int64

##### There is a tie for max salary, as the question is asking the next pay level (descending), hence we will consider the second highest distinct non null salary (198000) and not the second highest row (250000)

### A) The largest value just after the maximum

In [47]:
salaries = data_clean["salary"].dropna()
sec_highest = salaries[salaries<salaries.max()].max()

print("The second highest salary is : ", sec_highest)

The second highest salary is :  198000


### B) Find distinct values then sort in Descending order

In [53]:
sec_highest_distinct = salaries.drop_duplicates().nlargest(2).iloc[-1]
print("The second highest salary is : ", sec_highest_distinct)

The second highest salary is :  198000


### C) Using dense rank

In [56]:
data_ranked = data_clean.assign(
    salary_rank = data_clean["salary"].rank(method = "dense", ascending = False)
)

details_sec_highest_earner = data_ranked.loc[
data_ranked["salary_rank"]==2,
["employee_id", "full_name", "title", "salary"]
]

print(" The second highest salary is: : \n", details_sec_highest_earner)

 The second highest salary is: : 
    employee_id    full_name           title  salary
1         4003  Anika Conti  Senior Manager  198000


## Discussion: Which method is better?
### 1) Approach A: The Fast & Cheap Way as it uses just Two Aggregate Passes using O(n) time complexity.
#### It’s the cheapest method on massive datasets because databases can easily make use of indexes for MAX() operations.
#### It is good to use it when we only need the numerical value itself and query performance / resource cost is the top priority on a massive table.

### 2) Approach B: The Clean & Readable Way using Unique + Sort in just O(ulogu) time complexity where u is distinct number of salaries.
####   For everyday scripts, exploratory data analysis, or Pandas pipelines where code readability and simplicity matter more than optimizing for very large scale tables.
####   It works great as long as there aren't too many different salary numbers to sort through.

### 3) Approach C: The method using Dense Rank Window Function in O(nlogn) time complexity.
####   When we need the associated metadata—like knowing who earns that second-highest salary (e.g., employee name, department, ID), or if we need flexibility to instantly look up any rank we want—like 3rd, 4th, or 10th highest—without rewriting the code from scratch.

### **Report to the Compensation Team**
#### 1) The Second highest distinct salary on the record is 198,000.
#### 2) **Data Formatting Issue:** This figure was originally stored as a text string with a comma ("198,000"), which quietly broke any standard mathematical calculations on the column. Instead of dropping this entry, I repaired the formatting. Without this fix, the reported figure would have incorrectly defaulted to 181,000 (or worse).
#### 3) **Handling Ties:** Two top-earning employees tied at 250,000. Because of this tie, "second-highest" is defined as the second-highest distinct pay amount rather than simply the second row in the list. This distinction should be clearly documented whenever this number is referenced in official reports.
#### 4) **Missing Data:** Five employees currently have no salary on record. They were excluded from this specific ranking calculation but remain in the primary dataset. We should complete these missing records prior to the upcoming pay review cycle.
#### 5) **Recommendation for Future Cycles:** Since this salary lookup is executed regularly every review cycle, I have documented three distinct computation methods in the notebook, along with guidelines on when to use each approach as our dataset scales.